# eBird API
## MAD 2502 — Applied Data

**Big idea:** An API lets one program request structured data from another service.

This notebook starts with an invented offline example, so **you do not need an API key to learn the lesson**.

By the end, you should be able to explain:

- API, endpoint, request, response, and JSON;
- Python lists and dictionaries;
- API keys, headers, and parameters;
- how API records become rows in a DataFrame;
- the difference between **observation records** and **reported bird counts**;
- why a missing `howMany` value is not the same as zero.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect

[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/01_ebird_api.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/01_ebird_api.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/01_ebird_api.ipynb)

Run cells from the top with **Shift+Enter**. The sample lesson works without internet access after the packages are installed. The optional live request requires internet access and your own eBird API key.

In [ ]:
# If an import fails, uncomment and run this once.
# %pip install pandas requests

## 1. Understand — Asking another program for data

An **API** (application programming interface) is a documented way for programs to communicate.

| Term | Beginner meaning |
|---|---|
| **endpoint** | the web address for a particular API operation |
| **request** | the message your program sends to ask for data |
| **response** | what the service sends back |
| **JSON** | a common text format for structured data |
| **API key** | a private value that identifies authorized access |
| **header** | extra request information, such as an API key |
| **parameter** | an option sent with a request, such as number of days |

We will use eBird's recent-observations endpoint for one species:

```text
https://api.ebird.org/v2/data/obs/{region_code}/recent/{species_code}
```

`US-FL` is the region code for Florida. eBird species codes include `norcar` for Northern Cardinal, `blujay` for Blue Jay, and `amecro` for American Crow.

The parameters used here are:

- `back` — number of days to look back, from 1 to 30;
- `maxResults` — maximum number of records returned.

A JSON **array** becomes a Python **list**. A JSON **object** becomes a Python **dictionary**. JSON `null` becomes Python `None`.

A typical eBird response is therefore a **list of dictionaries**.

### API key safety

Never place a real API key in a notebook cell that may be committed or shared.

For an optional live request, copy `templates/my_api_info_example.py` to a file named `my_api_info.py` beside the notebook and add your own key there:

```python
# my_api_info.py — never commit this file
ebird_api_key = "YOUR_KEY_HERE"
```

In Colab, upload `my_api_info.py` privately to the session Files pane. It disappears when the session ends.

`.gitignore` helps prevent accidental Git commits, but it does **not** encrypt a key or protect a key that has already been committed.

In [ ]:
import json

import pandas as pd
import requests
from IPython.display import display

RUN_LIVE = False  # Leave False while learning from the sample.
API_KEY = None

if RUN_LIVE:
    try:
        from my_api_info import ebird_api_key
        API_KEY = ebird_api_key
    except ImportError:
        print("No key file found. The offline sample still works.")

## 2. Predict — Anatomy of a request

Before running the next cell, predict the exact endpoint URL.

Also predict where each item belongs:

- `region_code`
- `species_code`
- `days`
- the API key

The region and species are part of the endpoint path. `days` is a parameter. The API key is sent in a header.

In [ ]:
region_code = "US-FL"
species_code = "norcar"
days = 7

endpoint = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{species_code}"
)

params = {
    "back": days,
    "maxResults": 5000,
}

headers = {
    "X-eBirdApiToken": API_KEY,
}

print("endpoint:", endpoint)
print("parameters:", params)
print("API key configured:", API_KEY is not None)

The `headers` dictionary contains the API key only when you have configured one. We do **not** print the key.

When `requests.get(...)` is eventually called, it sends the endpoint, headers, and parameters together as one HTTP request.

## 3. Run — Start with an offline JSON response

The next cell uses invented teaching data, not live eBird observations.

`json.loads(...)` converts JSON text into Python values. `[0]` selects the first list item. `["locName"]` selects a dictionary value by key.

One returned dictionary becomes one DataFrame row.

In [ ]:
sample_json = """[
  {"speciesCode": "norcar", "locName": "Campus Garden", "howMany": 2},
  {"speciesCode": "norcar", "locName": "Lake Trail", "howMany": 5},
  {"speciesCode": "norcar", "locName": "Wetlands"}
]"""

practice_records = json.loads(sample_json)

print("response type:", type(practice_records))
print("first record type:", type(practice_records[0]))
print("first location:", practice_records[0]["locName"])

practice_birds = pd.DataFrame(practice_records)
display(practice_birds)

### Observation records are not the same as bird counts

This distinction is essential:

- `len(practice_birds)` counts **returned observation records**.
- `howMany` is the reported number of individual birds for a record, when that field is present.
- adding `howMany` values gives a total of the **known reported counts in these returned records**.

That total is **not** a population estimate and does not necessarily represent unique birds. Different reports can refer to the same birds, and result limits can omit records.

The third sample record has no `howMany` key. pandas displays that missing value as `NaN`. Missing means **unknown**, not zero.

In [ ]:
counts = pd.to_numeric(
    practice_birds["howMany"],
    errors="coerce",
)

print("observation records returned:", len(practice_birds))
print("known reported bird total:", counts.sum(min_count=1))
print("records with missing howMany:", counts.isna().sum())

`pd.to_numeric(..., errors="coerce")` tries to make the values numeric. A value that cannot be converted becomes missing.

`sum(min_count=1)` is useful here because if **all** counts were missing, the result would remain missing instead of being reported as zero.

## 4. Change — Change one part without changing later cells

Change one thing at a time and predict the effect.

The code below uses **new variable names** so this experiment does not overwrite the original `species_code` and `days` used later.

Try:

1. `changed_species_code = "blujay"`
2. `changed_days = 10`
3. one more pretend record with `howMany = 12`

In [ ]:
changed_species_code = "blujay"
changed_days = 10

changed_endpoint = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{changed_species_code}"
)

changed_params = {
    "back": changed_days,
    "maxResults": 5000,
}

print("changed endpoint:", changed_endpoint)
print("changed parameters:", changed_params)

## 5. Use — Make the optional live request

An HTTP **status code** reports how the request went.

- `200` usually means success.
- `400` means the request was not acceptable.
- `401` or `403` often indicates an authentication/access problem.
- `429` means too many requests.
- `5xx` indicates a server-side problem.

`raise_for_status()` stops the analysis when the server returns an HTTP error. The timeout prevents the request from waiting forever.

The function below combines the request pieces. You do not need to memorize the function yet; focus on how the endpoint, header, parameters, response, and JSON fit together.

In [ ]:
def get_ebird_data(
    api_key,
    region_code,
    species_code,
    days=7,
    max_results=5000,
):
    """Return recent eBird records as a DataFrame."""

    if not api_key or api_key == "YOUR_KEY_HERE":
        raise ValueError("Add your own key in my_api_info.py first.")

    if not 1 <= days <= 30:
        raise ValueError("days must be between 1 and 30.")

    endpoint = (
        "https://api.ebird.org/v2/data/obs/"
        f"{region_code}/recent/{species_code}"
    )

    response = requests.get(
        endpoint,
        headers={"X-eBirdApiToken": api_key},
        params={
            "back": days,
            "maxResults": max_results,
        },
        timeout=30,
    )

    response.raise_for_status()
    records = response.json()

    if not isinstance(records, list):
        raise ValueError("Expected the API response to be a list.")

    if not all(isinstance(record, dict) for record in records):
        raise ValueError("Expected every returned record to be a dictionary.")

    data = pd.DataFrame(records)

    for column in ["speciesCode", "locName", "howMany"]:
        if column not in data.columns:
            data[column] = pd.NA

    data["howMany"] = pd.to_numeric(
        data["howMany"],
        errors="coerce",
    )

    if len(data) == max_results:
        print("Result cap reached; this may not be a complete set of records.")

    return data

In [ ]:
birds = practice_birds.copy()

if RUN_LIVE:
    if not API_KEY or API_KEY == "YOUR_KEY_HERE":
        print("Using teaching sample: configure your key before requesting live data.")
    else:
        try:
            birds = get_ebird_data(
                API_KEY,
                region_code,
                species_code,
                days=days,
            )
            print("Source: live eBird response")
        except (requests.RequestException, ValueError) as error:
            status = getattr(
                getattr(error, "response", None),
                "status_code",
                None,
            )
            print(
                "Request failed:",
                type(error).__name__,
                "HTTP status:",
                status,
            )
            print("Source: teaching sample; no live result is being reported.")
else:
    print("Source: invented teaching sample (RUN_LIVE is False)")

display(birds.head())

bird_counts = pd.to_numeric(
    birds["howMany"],
    errors="coerce",
)

print("observation records returned:", len(birds))
print("known reported bird total:", bird_counts.sum(min_count=1))
print("records with missing howMany:", bird_counts.isna().sum())

When interpreting live results, keep the questions separate:

1. **How many observation records were returned?** Use the number of rows.
2. **What is the sum of known reported bird counts?** Add the nonmissing `howMany` values.
3. **How many records have unknown counts?** Count missing `howMany` values.

None of these numbers is automatically a population census.

## 6. Practice

1. In your own words, define **API**, **endpoint**, **request**, and **response**.
2. Explain why the API key belongs in a header rather than in a committed notebook.
3. Change one sample `howMany` value and predict the new known reported total.
4. Count sample observation records and known reported birds separately.
5. Explain what the missing `howMany` value means.
6. Change the region/species variables and print the new endpoint without making a request.
7. Optional, with your own key: request Northern Cardinal (`norcar`) records for Florida (`US-FL`) for 7 days, then change only the species to Blue Jay (`blujay`).
8. For a live request, record the date, parameters, and result limit in your notes.

## 7. Connect

The same pattern appears in many APIs:

**build a request → receive a response → convert JSON to Python objects → build a table → ask a question**

For a capstone API source, record:

- what one returned record represents;
- the endpoint and important parameters;
- authentication requirements;
- result limits or caps;
- missing fields;
- enough information to reproduce the request later.

**Next connection:** pandas Basics focuses on inspecting, filtering, and summarizing the DataFrame after data have been collected.

[eBird API documentation](https://documenter.getpostman.com/view/664302/S1ENwy59) · [Request an eBird API key](https://ebird.org/api/keygen)